In [ ]:
# 30. CBOW using a Sample Text Corpus 
# Develop a CBOW model to learn word representations from a given text 
# corpus. Prepare the text data, generate context-target training pairs using 
# a fixed context window, train an embedding-based neural network and 
# predict the target word for given context words. 

In [1]:
# Implement CBOW
# Step 1: Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import re

In [2]:
# --------------------------------
# a. DATA PREPARATION
# --------------------------------

# Step 2: Create sample text corpus
text = """
the black cat sits on the warm mat
the brown dog sits on the cold floor
the fluffy cat likes to chase the playful dog
the friendly dog likes to play with the cute cat
deep learning and natural language processing are fascinating
word embeddings capture semantic meanings of words
continuous bag of words model learns context efficiently
"""

In [3]:
# Step 3: Text preprocessing and tokenization

# Convert text into lowercase and remove punctuation
text = re.sub(r'[^\w\s]', '', text.lower())

# Tokenize text into words
words = text.split()

print("Tokenized Words:")
print(words)

Tokenized Words:
['the', 'black', 'cat', 'sits', 'on', 'the', 'warm', 'mat', 'the', 'brown', 'dog', 'sits', 'on', 'the', 'cold', 'floor', 'the', 'fluffy', 'cat', 'likes', 'to', 'chase', 'the', 'playful', 'dog', 'the', 'friendly', 'dog', 'likes', 'to', 'play', 'with', 'the', 'cute', 'cat', 'deep', 'learning', 'and', 'natural', 'language', 'processing', 'are', 'fascinating', 'word', 'embeddings', 'capture', 'semantic', 'meanings', 'of', 'words', 'continuous', 'bag', 'of', 'words', 'model', 'learns', 'context', 'efficiently']


In [4]:
# --------------------------------
# b. GENERATE TRAINING DATA
# --------------------------------

# Step 4. Create vocabulary

vocab = sorted(set(words))
vocab_size = len(vocab)

print("\nVocabulary:")
print(vocab)

print("\nVocabulary Size:", vocab_size)


Vocabulary:
['and', 'are', 'bag', 'black', 'brown', 'capture', 'cat', 'chase', 'cold', 'context', 'continuous', 'cute', 'deep', 'dog', 'efficiently', 'embeddings', 'fascinating', 'floor', 'fluffy', 'friendly', 'language', 'learning', 'learns', 'likes', 'mat', 'meanings', 'model', 'natural', 'of', 'on', 'play', 'playful', 'processing', 'semantic', 'sits', 'the', 'to', 'warm', 'with', 'word', 'words']

Vocabulary Size: 41


In [5]:
# Step 5. Assign unique indices to words

# Assign a unique index to every word
word_to_index = {
    word: i for i, word in enumerate(vocab)
}
# Reverse mapping: index to word
index_to_word = {
    i: word for word, i in word_to_index.items()
}
print("\nWord to Index:")
print(word_to_index)


Word to Index:
{'and': 0, 'are': 1, 'bag': 2, 'black': 3, 'brown': 4, 'capture': 5, 'cat': 6, 'chase': 7, 'cold': 8, 'context': 9, 'continuous': 10, 'cute': 11, 'deep': 12, 'dog': 13, 'efficiently': 14, 'embeddings': 15, 'fascinating': 16, 'floor': 17, 'fluffy': 18, 'friendly': 19, 'language': 20, 'learning': 21, 'learns': 22, 'likes': 23, 'mat': 24, 'meanings': 25, 'model': 26, 'natural': 27, 'of': 28, 'on': 29, 'play': 30, 'playful': 31, 'processing': 32, 'semantic': 33, 'sits': 34, 'the': 35, 'to': 36, 'warm': 37, 'with': 38, 'word': 39, 'words': 40}


In [6]:
# Step 6. Generate context-target pairs
# Define context window size
window_size = 2

contexts = []
targets = []

# Generate context-target pairs
for i in range(window_size, len(words) - window_size):

    context = []

    for j in range(i - window_size, i + window_size + 1):

        if j != i:
            context.append(word_to_index[words[j]])

    contexts.append(context)
    targets.append(word_to_index[words[i]])
    
# Convert into NumPy arrays
contexts = np.array(contexts)
targets = np.array(targets)

print("\nTotal Training Samples:", len(contexts))


Total Training Samples: 54


In [7]:
# Step 7. Display sample training pairs
print("\nSample Context-Target Pairs:")

for i in range(5):
    context_words = [
        index_to_word[idx] for idx in contexts[i]
    ]

    print("Context:", context_words)
    print("Target:", index_to_word[targets[i]])
    print()



Sample Context-Target Pairs:
Context: ['the', 'black', 'sits', 'on']
Target: cat

Context: ['black', 'cat', 'on', 'the']
Target: sits

Context: ['cat', 'sits', 'the', 'warm']
Target: on

Context: ['sits', 'on', 'warm', 'mat']
Target: the

Context: ['on', 'the', 'mat', 'the']
Target: warm



In [8]:
# --------------------------------
# c. BUILD AND TRAIN CBOW MODEL
# --------------------------------

# Step 8. Build CBOW neural network
model = keras.Sequential([
    layers.Input(shape=(window_size * 2,)),
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=50
    ),
    layers.GlobalAveragePooling1D(),
    layers.Dense(vocab_size, activation="softmax")
])

In [9]:
# Step 9. Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
# Step 10. Train the model
history = model.fit(
    contexts,
    targets,
    epochs=1000,
    verbose=0
)

print("\nModel Training Completed!")


Model Training Completed!


In [11]:
# --------------------------------
# d. OUTPUT AND EVALUATION
# --------------------------------

# Step 11. Evaluate the model
loss, accuracy = model.evaluate(
    contexts,
    targets,
    verbose=0
)
print("\nTraining Loss:", loss)
print("Training Accuracy:", accuracy)


Training Loss: 0.0005943879950791597
Training Accuracy: 1.0


In [12]:
# Step 12. Predict the target word
sample_context = ["the", "black", "sits", "on"]

# Convert context words into indices
sample = np.array([

    [word_to_index[word] for word in sample_context]

])

prediction = model.predict(sample, verbose=0)

predicted_index = np.argmax(prediction)

print("\nGiven Context:", sample_context)

print("Actual Target Word:",
      index_to_word[targets[0]])

print("Predicted Target Word:",
      index_to_word[predicted_index])



Given Context: ['the', 'black', 'sits', 'on']
Actual Target Word: cat
Predicted Target Word: cat
